# ⚡ EnerVision AI
## Smart Energy Forecasting & Automated Demand Management System

### Project Overview
EnerVision AI is an AI-powered energy management system designed to forecast the hourly Net Energy Balance in smart residential buildings up to 24 hours in advance.
The system analyzes historical energy consumption, solar power generation, weather conditions, and time-based features to predict whether the building will experience an energy surplus or shortage.

$$\text{Net Energy Balance} = \text{Solar Power Generation} - \text{Building Energy Demand}$$

### Benchmarked Models:
1. **PatchTSMixer:** Primary state-of-the-art (SOTA) deep learning model for multivariate time-series forecasting.
2. **Bi-LSTM:** Recurrent deep learning architecture for temporal sequences.
3. **XGBoost Regressor:** Fast gradient-boosted machine learning baseline.

### Operational Statuses:
- 🟢 **Surplus:** Net > +1.5 kW
- 🟡 **Balance:** 0 kW <= Net <= +1.5 kW
- 🔴 **Shortage:** Net < 0 kW

#1. Import libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler




# Plot style configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# 2. Load Data and Run Sanity Checks

In [ ]:
# Mount Drive and load dataset
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/EnerVision_AI/Data/HomeC.csv'
df = pd.read_csv(DATA_PATH, encoding='latin1', on_bad_lines='skip', engine='python')
print("Raw Dataset Shape:", df.shape)
df.head()

## 2.1 Shape and dtypes

In [ ]:
# Sanity Checks: Clean timestamp and calculate target metrics
df = df[df['time'].astype(str).str.strip() != "\\"].copy()

df['time'] = pd.to_numeric(df['time'], errors='coerce')
df = df.dropna(subset=['time'])

start_time = pd.to_datetime(int(df['time'].iloc[0]), unit='s')
df['time'] = pd.date_range(start=start_time, periods=len(df), freq='min')

df['Demand_kW'] = pd.to_numeric(df['use [kW]'], errors='coerce')
solar_col = 'gen [kW]' if 'gen [kW]' in df.columns else 'Solar [kW]'
df['Solar_kW'] = pd.to_numeric(df[solar_col], errors='coerce')

df['Net_Energy'] = df['Solar_kW'] - df['Demand_kW']

print("Cleaned Dataset Shape:", df.shape)
print("Duplicate Rows:", df.duplicated().sum())
print("Missing Values in Target:", df['Net_Energy'].isna().sum())

# 3. EDA: understand the data

### 3.1 Descriptive Statistics

In [ ]:
key_vars = ['Solar_kW', 'Demand_kW', 'Net_Energy', 'temperature', 'cloudCover', 'humidity', 'windSpeed']
valid_key_vars = [col for col in key_vars if col in df.columns]
df[valid_key_vars].describe().T

### 3.2 24-Hour Diurnal Energy Profile

In [ ]:
df['hour'] = df['time'].dt.hour
hourly_profile = df.groupby('hour')[['Solar_kW', 'Demand_kW', 'Net_Energy']].mean()

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.plot(hourly_profile.index, hourly_profile['Solar_kW'], label='Solar Generation (kW)', color='#2ca02c', linewidth=2.2)
ax.plot(hourly_profile.index, hourly_profile['Demand_kW'], label='Building Demand (kW)', color='#d62728', linewidth=2.2, linestyle='--')
ax.plot(hourly_profile.index, hourly_profile['Net_Energy'], label='Net Energy Balance (kW)', color='#1f77b4', linewidth=2.5)

ax.fill_between(hourly_profile.index, hourly_profile['Solar_kW'], hourly_profile['Demand_kW'],
                where=(hourly_profile['Solar_kW'] >= hourly_profile['Demand_kW']), color='#2ca02c', alpha=0.2, label='Surplus Window')
ax.fill_between(hourly_profile.index, hourly_profile['Solar_kW'], hourly_profile['Demand_kW'],
                where=(hourly_profile['Solar_kW'] < hourly_profile['Demand_kW']), color='#d62728', alpha=0.15, label='Deficit Window')

ax.set_title('Diurnal Energy Profile: Solar vs. Demand', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Hour of Day (0-23)', fontsize=10)
ax.set_ylabel('Power (kW)', fontsize=10)
ax.set_xticks(range(0, 24))
ax.legend(frameon=True, facecolor='white', loc='upper right')
plt.tight_layout()
plt.show()

### 3.3 Meteorological Correlation Heatmap

In [ ]:
weather_vars = ['temperature', 'cloudCover', 'humidity', 'windSpeed', 'Solar_kW', 'Demand_kW', 'Net_Energy']
valid_weather_vars = [col for col in weather_vars if col in df.columns]

# Convert 'cloudCover' to numeric, coercing errors
if 'cloudCover' in df.columns:
    df['cloudCover'] = pd.to_numeric(df['cloudCover'], errors='coerce')
    # Fill NaN values with the mean of the column to allow correlation calculation
    df['cloudCover'].fillna(df['cloudCover'].mean(), inplace=True)

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
sns.heatmap(df[valid_weather_vars].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            annot_kws={"size": 9, "weight": "bold"}, linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Meteorological & Energy Correlation Matrix', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### 3.4 Net Energy Balance Distribution & Operational Thresholds


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
sns.histplot(df['Net_Energy'], kde=True, bins=60, color='#1f77b4', stat='density', alpha=0.35, ax=ax)

ax.axvline(1.5, color='green', linestyle='--', linewidth=2, label='Surplus Threshold (🟢 > +1.5 kW)')
ax.axvline(0.0, color='red', linestyle='--', linewidth=2, label='Shortage Threshold (🔴 < 0.0 kW)')

ax.set_title('Net Energy Differential Distribution & Decision Engine Thresholds', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Net Energy Balance (kW)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.legend(frameon=True, facecolor='white', loc='upper right')
plt.tight_layout()
plt.show()

# 4. Feature Engineering

### 4.1. Temporal factors and identifying weekends

In [ ]:
# 1. Temporal factors and identifying weekends
df['month'] = df['time'].dt.month
df['dayofweek'] = df['time'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)




###4.2. Transforming time into cyclical features (Sine/Cosine)

In [ ]:
# 2. Transforming time into cyclical features (Sine/Cosine)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)


###4.3. Adding lag features

In [ ]:
# 3. Adding lag features
# Net_Energy lags: kept for the REAL-TIME status classifier (Surplus/Balance/
# Shortage), which is computed from the ACTUAL current Solar & Demand values,
# not from a model prediction.
df['Net_Energy_lag1'] = df['Net_Energy'].shift(1)
df['Net_Energy_lag60'] = df['Net_Energy'].shift(60)
df['Net_Energy_lag1440'] = df['Net_Energy'].shift(1440)

# Demand lags: NEW - added so the forecasting models (XGBoost, Bi-LSTM,
# PatchTSMixer) can predict future household Consumption/Demand directly,
# using its own past values as a feature, instead of predicting Net_Energy.
df['Demand_lag1'] = df['Demand_kW'].shift(1)
df['Demand_lag60'] = df['Demand_kW'].shift(60)
df['Demand_lag1440'] = df['Demand_kW'].shift(1440)


###4.4 Dropping empty rows




In [ ]:
# 4. Dropping empty rows resulting from the shift operation
df = df.dropna().reset_index(drop=True)

In [ ]:
print("Shape after Feature Engineering:", df.shape)

# 5. Saveing the clean data.

In [ ]:
df.to_csv('/content/drive/MyDrive/EnerVision_AI/Data/HomecCleaned.csv', index=False)
print("✅ Cleaned dataset successfully saved to Google Drive!")

#...